# Perfilar RUTs — prueba manual

Notebook de inspección para #156. Toma un RUT, lo pasa por una de las tres
fuentes (Web Empresario, ruts.info, SRE) y muestra el borrador de perfil
resultante: nombre, vigencia, región, **rubros** y **palabras clave**
detectados a partir de las actividades económicas.

No reemplaza `perfilar_rut.py` (la CLI ya existente) — es la misma lógica,
pero cómoda para revisar varios RUTs seguidos y comparar la salida a ojo,
que es justo lo que hace falta para la prueba pendiente con los 5-6 RUTs
reales (ver `1.1-onboarding.md`, sección "Prueba pendiente con RUTs reales").

**No hace ninguna llamada de red por su cuenta.** Cada fuente necesita su
propia URL y token, configurados en un archivo `.env` — ver la celda de
configuración más abajo — o, si no se configuran, se puede cargar un JSON
ya guardado con `payload_path`.


In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

from perfilar_rut import FUENTES, a_salida, consultar_api, normalizar_rut
from perfilamiento.perfil import PerfilSugerido

print("Fuentes disponibles:", list(FUENTES))


## Configuración

Copiar `.env.example` a `.env` (en esta misma carpeta) y completar con las
credenciales reales. Se carga solo al importar `perfilar_rut` (celda
siguiente), sin importar si este notebook se corre desde VS Code, JupyterLab
o la terminal — no hace falta exportar nada a mano.

```bash
# poc/.env.example
WEB_EMPRESARIO_URL=https://.../v1/{rut}
WEB_EMPRESARIO_TOKEN=

RUTS_INFO_URL=https://.../consulta?rut={rut}
RUTS_INFO_TOKEN=

SRE_URL=https://sre.cl/api/company_info
SRE_TOKEN=
```

`.env` está en `.gitignore` — nunca se commitea. Sin él (o con una fuente sin
completar), `perfilar_uno(..., fuente=...)` con un RUT en vez de un payload va
a fallar con un mensaje claro (`Falta WEB_EMPRESARIO_URL...`), no con un error
críptico.


In [ ]:
import json


def perfilar_uno(*, fuente: str, rut: str | None = None, payload_path: str | None = None) -> PerfilSugerido:
    """Perfila un RUT: desde la API (si `rut`) o desde un JSON guardado (si `payload_path`).

    Usar `payload_path` para no gastar cuota de la API mientras se prueba el
    notebook, o para repetir exactamente la misma respuesta sin volver a
    consultar.
    """
    fuente_cfg = FUENTES[fuente]

    if payload_path:
        payload = json.loads(Path(payload_path).read_text(encoding="utf-8"))
    elif rut:
        rut_normalizado = normalizar_rut(rut)
        payload = consultar_api(fuente_cfg, fuente, rut_normalizado)
    else:
        raise ValueError("hay que pasar `rut` o `payload_path`")

    return fuente_cfg.interpretar(payload)


## Un RUT a la vez

Cambiar `FUENTE` y `RUT` acá abajo. `FUENTE` es una de `"web-empresario"`,
`"ruts-info"` o `"sre"`.


In [ ]:
FUENTE = "web-empresario"
RUT = "76.668.304-5"

perfil = perfilar_uno(fuente=FUENTE, rut=RUT)
salida = a_salida(perfil)
salida


### Detalle legible

Lo mismo que imprime `perfilar_rut.py` en modo texto, para no tener que leer
el diccionario crudo.


In [ ]:
from perfilar_rut import imprimir

imprimir(perfil)


## Varios RUTs seguidos (la prueba de #156)

Acá van los 5-6 RUTs reales cuando estén disponibles. Cada fila es
`(fuente, rut)`; se puede mezclar fuentes distintas para la misma empresa y
comparar qué entrega cada una (ver "Comparación de cobertura" en
`1.1-onboarding.md`).

**Pendiente:** reemplazar la lista de ejemplo por los RUTs reales que se
van a usar para la prueba.


In [ ]:
RUTS_A_PROBAR = [
    ("web-empresario", "76.668.304-5"),  # ejemplo ya usado en la documentación
    # ("web-empresario", "TODO"),
    # ("web-empresario", "TODO"),
    # ("web-empresario", "TODO"),
    # ("web-empresario", "TODO"),
]

resultados = []
for fuente, rut in RUTS_A_PROBAR:
    try:
        perfil = perfilar_uno(fuente=fuente, rut=rut)
        resultados.append(a_salida(perfil))
    except SystemExit as exc:
        # `consultar_api` corta con SystemExit si falta la URL/token o si la
        # API respondió con error — se captura acá para que un RUT que falla
        # no interrumpa el resto de la lista.
        resultados.append({"rut": rut, "fuente": fuente, "error": str(exc)})

resultados


### Tabla comparativa

Una fila por empresa, para revisar de un vistazo si el rubro y las palabras
clave tienen sentido. Sin `pandas` a propósito — no vale la pena la
dependencia para esto — se arma como tabla HTML simple.


In [ ]:
from IPython.display import HTML


def tabla_html(filas: list[dict]) -> HTML:
    columnas = ["nombre", "rut", "fuente", "vigente", "regiones", "sectores", "palabras_clave", "avisos"]
    encabezado = "".join(f"<th style='text-align:left'>{c}</th>" for c in columnas)
    cuerpo = ""
    for fila in filas:
        if "error" in fila:
            cuerpo += f"<tr><td colspan='{len(columnas)}'>{fila['rut']} ({fila['fuente']}): {fila['error']}</td></tr>"
            continue
        celdas = []
        for c in columnas:
            valor = fila.get(c, "")
            if isinstance(valor, list):
                valor = ", ".join(str(v) for v in valor) or "—"
            celdas.append(f"<td style='vertical-align:top'>{valor}</td>")
        cuerpo += "<tr>" + "".join(celdas) + "</tr>"
    return HTML(f"<table><tr>{encabezado}</tr>{cuerpo}</table>")


tabla_html(resultados)


## Sin red: cargar un payload ya guardado

Para repetir un caso sin consultar la API de nuevo (o para revisar un
payload que se guardó a mano), usar `payload_path` en vez de `rut`:

```python
perfil = perfilar_uno(fuente="sre", payload_path="respuesta_sre_ejemplo.json")
```

Los payloads no se versionan si tienen datos de una empresa real — misma
regla que el resto de `corpus/`.
